In [1]:
import sys; sys.path.append("../src")
import numpy as np, pandas as pd
from pathlib import Path
from sklearn.ensemble import IsolationForest
from data import load, TIME, AMOUNT, SPLIT
from metrics import evaluate

tx, feats = load()
train_mask    = tx[TIME] <= SPLIT
test_lab_mask = (tx[TIME] > SPLIT) & tx["y"].notna()

X = tx[feats].fillna(tx.loc[train_mask, feats].median())   # fill gaps using train only
y = tx.loc[test_lab_mask, "y"].to_numpy()
print(len(feats), "features | train rows:", train_mask.sum())

182 features | train rows: 136265


In [2]:
iso = IsolationForest(n_estimators=200, random_state=42, n_jobs=-1)
iso.fit(X[train_mask])                      # all train rows, labels never used

tx["iso_score"] = -iso.score_samples(X)     # flip sign: higher = more anomalous
thr = tx.loc[train_mask, "iso_score"].quantile(0.99)

amt = tx[AMOUNT].fillna(0)
rng = np.random.default_rng(42)
results = [
    evaluate("Random (reference)", y, rng.random(len(y)), 0.99),
    evaluate("Amount baseline", y, amt[test_lab_mask].to_numpy(),
             amt[train_mask].quantile(0.99)),
    evaluate("Isolation Forest (all features)", y,
             tx.loc[test_lab_mask, "iso_score"].to_numpy(), thr),
]
pd.DataFrame(results)

,model,P@50,P@100,P@500,PR-AUC,flag_rate_%
0,Random (reference),0.02,0.05,0.07,0.066,1.14
1,Amount baseline,0.00,0.00,0.00,0.044,1.19
2,Isolation Forest (all features),0.00,0.00,0.00,0.037,6.10


In [3]:
local = [c for c in feats if c.lower().startswith("local")]
print(len(local), "local features")

iso_l = IsolationForest(n_estimators=200, random_state=42, n_jobs=-1)
iso_l.fit(X.loc[train_mask, local])
tx["iso_local"] = -iso_l.score_samples(X[local])

results.append(evaluate("Isolation Forest (local only)", y,
               tx.loc[test_lab_mask, "iso_local"].to_numpy(),
               tx.loc[train_mask, "iso_local"].quantile(0.99)))
pd.DataFrame(results)

93 local features


,model,P@50,P@100,P@500,PR-AUC,flag_rate_%
0,Random (reference),0.02,0.05,0.07,0.066,1.14
1,Amount baseline,0.00,0.00,0.00,0.044,1.19
2,Isolation Forest (all features),0.00,0.00,0.00,0.037,6.10
3,Isolation Forest (local only),0.00,0.00,0.00,0.038,4.52


In [5]:
best = "iso_score"   # change to "iso_local" if that one won
Path("../results").mkdir(exist_ok=True)
tx[["txId", TIME, AMOUNT, "y", best]].rename(columns={best: "score"}) \
  .to_csv("../results/scores.csv", index=False)
pd.DataFrame(results).to_csv("../results/metrics.csv", index=False)